[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nlavadodatasci/atd-megaf-fiabilidade/blob/main/T6_diagramas_de_blocos/T6_diagramas_de_blocos.ipynb)

# T6 — Fiabilidade de sistemas: diagramas de blocos

**ATD — MEGAF 2026/27 · Atividade de Fiabilidade · Tema T6**

Como se passa da fiabilidade de cada componente para a fiabilidade do sistema? Este *notebook* define três funções (série, paralelo e $k$-de-$n$) e aplica-as a um sistema **inventado**. O sistema da instalação fica para o grupo.

**Como usar:** execute as células por ordem (Shift+Enter). As primeiras secções estão resolvidas, com dados da aula ou com exemplos inventados, para mostrar o método. A última secção, **A sua vez**, lê os dados da instalação e deixa pistas: o trabalho do grupo começa aí. O capítulo deste tema no documento de apoio explica cada passo.

## 1. Três funções

- **Série**: o sistema funciona se **todos** os blocos funcionarem: $R = R_1 \times R_2 \times \dots$
- **Paralelo**: o sistema só falha se **todos** os blocos falharem: $R = 1 - (1-R_1)(1-R_2)\dots$
- **$k$-de-$n$**: $n$ blocos iguais, o sistema funciona se pelo menos $k$ funcionarem. É uma conta de distribuição binomial.

As fórmulas pressupõem que os blocos falham de forma **independente**.

In [1]:
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

REPO = "https://raw.githubusercontent.com/nlavadodatasci/atd-megaf-fiabilidade/main/dados/"

def obter(ficheiro):
    """Vai buscar o ficheiro ao repositório. Se falhar, abre uma janela
    para o escolher no seu computador."""
    if not os.path.exists(ficheiro):
        try:
            urllib.request.urlretrieve(REPO + ficheiro, ficheiro)
        except Exception:
            from google.colab import files
            print("Escolha no seu computador o ficheiro", ficheiro)
            carregados = files.upload()
            if ficheiro not in carregados:               # veio com outro nome: renomeia
                os.rename(list(carregados)[0], ficheiro)
    return ficheiro

def serie(*R):
    return np.prod(R)

def paralelo(*R):
    return 1 - np.prod([1 - r for r in R])

def k_de_n(k, n, R):
    # probabilidade de pelo menos k dos n blocos (iguais, fiabilidade R) funcionarem
    return stats.binom.sf(k - 1, n, R)

print("Série de dois blocos 0,90:     ", serie(0.90, 0.90))
print("Paralelo de dois blocos 0,90:  ", paralelo(0.90, 0.90))
print("2-de-3 com blocos 0,90:        ", k_de_n(2, 3, 0.90))

Série de dois blocos 0,90:      0.81
Paralelo de dois blocos 0,90:   0.99
2-de-3 com blocos 0,90:         0.972


Duas verificações que servem sempre: a fiabilidade de uma série é **menor ou igual** à do pior bloco; a de um paralelo é **maior ou igual** à do melhor.

E dois casos particulares do $k$-de-$n$: 1-de-$n$ é um paralelo; $n$-de-$n$ é uma série.

In [2]:
print(k_de_n(1, 3, 0.90), "=", paralelo(0.90, 0.90, 0.90))
print(k_de_n(3, 3, 0.90), "=", serie(0.90, 0.90, 0.90))

0.999 = 0.999
0.7290000000000001 = 0.7290000000000001


## 2. Um sistema misto inventado

Uma estação de bombagem: duas linhas em paralelo, cada uma com um motor e uma bomba em série, seguidas de uma válvula comum.

```
        +-- Motor --- Bomba --+
 ---o---|                     |---o--- Válvula ---
        +-- Motor --- Bomba --+
```

Fiabilidades inventadas para uma certa missão: motor 0,92; bomba 0,90; válvula 0,99.

O método é reduzir o diagrama **de dentro para fora**: primeiro cada linha (série), depois o conjunto das duas linhas (paralelo), por fim a série com a válvula.

In [3]:
R_motor, R_bomba, R_valvula = 0.92, 0.90, 0.99

linha    = serie(R_motor, R_bomba)             # motor e bomba em série
linhas   = paralelo(linha, linha)              # duas linhas em paralelo
estacao  = serie(linhas, R_valvula)            # tudo em série com a válvula

print(f"Uma linha:        {linha:.4f}")
print(f"Duas linhas:      {linhas:.4f}")
print(f"Estação completa: {estacao:.4f}")

Uma linha:        0.8280
Duas linhas:      0.9704
Estação completa: 0.9607


## 3. De onde vêm as fiabilidades dos blocos?

No documento base, as fiabilidades da Fig. 17 são dadas. Na instalação, tem de as calcular a partir dos tempos até à falha, para o tempo de missão. Há duas formas, mostradas aqui com os dados da aula e uma missão de 2000 h:

In [4]:
aula = pd.read_excel(obter("tempo_falhas_aula.xlsx"), sheet_name="tempo_falhas", usecols="A:B")
t = aula.iloc[:, 1].dropna().to_numpy()      # tempos até à falha da aula (horas)
print("Número de tempos:", len(t), "  MTBF:", round(t.mean(), 2))

missao = 2000
print(f"Proporção de unidades que durou mais de {missao} h: {(t > missao).mean():.4f}")
print(f"Pelo modelo exponencial ajustado:               {stats.expon(scale=t.mean()).sf(missao):.4f}")

Número de tempos: 9998   MTBF: 8738.14
Proporção de unidades que durou mais de 2000 h: 0.7935
Pelo modelo exponencial ajustado:               0.7954


Com quase 10 000 tempos, as duas formas dão praticamente o mesmo. Com 50 unidades por componente, a proporção só pode tomar valores de 2 em 2 % e é muito imprecisa quando a fiabilidade é alta. É por isso que se ajusta um modelo. Qual modelo usar para cada componente é a pergunta do tema T3: fale com esse grupo, ou use o *notebook* `00_comum`.

## 4. A sua vez

Pistas:

1. Resolva primeiro o estudo de caso do documento base, com os valores da Fig. 17. Se as suas funções reproduzirem a resolução à mão, pode confiar nelas.
2. Calcule $R(2000)$ de cada um dos cinco componentes da instalação.
3. Reduza o diagrama da instalação de dentro para fora (a secção "Os dados da instalação" do documento de apoio descreve a estrutura). A rede e os geradores são alternativas uma à outra: como se combinam?
4. Qual é o bloco que mais limita a fiabilidade do sistema? Porquê?
5. Variante: a carga exige dois dos três geradores. Que função usa, e quanto muda o resultado?
6. As fórmulas pressupõem falhas independentes. Dê um exemplo, nesta instalação, de uma causa que poderia fazer falhar vários blocos ao mesmo tempo.
7. Um gerador de emergência está parado até ser preciso (redundância em espera). Em que é que isso difere de um paralelo "normal"?

In [5]:
ensaios = pd.read_excel(obter("dados_fiabilidade.xlsx"), sheet_name="ensaios_vida")
missao = 2000

# Fiabilidade de cada componente para a missão (a preencher pelo grupo)
R = {"Interruptor": None, "Transformador": None, "QGBT": None, "Gerador": None, "Controlador": None}

# Primeiro passo da redução: um ramo da rede é um interruptor em série com um transformador
# ramo = serie(R["Interruptor"], R["Transformador"])
# rede = ...